In [2]:
import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.util.graphs import MARKERS, HATCHES, COLORBLIND_PALETTE, seaborn_setup, save_plot


## Phrase frequency


In [3]:
pefk = pd.read_csv("../../facilitation-dataset/pefk.csv")
seaborn_setup()


In [9]:
mod_df = pefk[pefk['is_moderator'] == True].copy().reset_index(drop=True)


def clean_text(text_input):
    if pd.isna(text_input):
        return ""

    text_input = text_input.lower()

    # fix glued words
    text_input = re.sub(r'regulationroom', 'regulation room', text_input)

    # remove urls
    text_input = re.sub(r'http\S+|www\S+', '', text_input)

    # keep letters only
    text_input = re.sub(r'[^a-z\s]', '', text_input)

    # collapse spaces
    text_input = re.sub(r'\s+', ' ', text_input).strip()

    return text_input


mod_df["clean_text"] = mod_df["text"].apply(clean_text)

# ---------------------------
# 3. Stopwords
# ---------------------------
custom_stopwords = list(text.ENGLISH_STOP_WORDS) + [
    'http', 'www', 'org', 'wiki', 'wikipedia', 'en'
]

# ---------------------------
# 4. TF-IDF
# ---------------------------
vectorizer = TfidfVectorizer(
    ngram_range=(2, 3),
    stop_words=custom_stopwords,
    min_df=2
)

X = vectorizer.fit_transform(mod_df["clean_text"])
feature_names = vectorizer.get_feature_names_out()


In [10]:
results = {}

for dataset, group in mod_df.groupby("dataset"):

    idx = group.index.to_numpy()
    X_subset = X[idx]

    # mean TF-IDF per phrase, within this dataset only
    scores = np.asarray(X_subset.mean(axis=0)).ravel()

    # A score of 0 here means the phrase never occurs in this dataset at
    # all (not "occurs weakly") -- it's the same kind of 0-as-missing case
    # as in the other notebook, so it must be excluded before ranking,
    # otherwise argsort() can pad the "top 10" with phrases that are
    # absent from the dataset whenever fewer than 10 phrases actually
    # occur in it.
    nonzero_idx = np.where(scores > 0)[0]

    if len(nonzero_idx) == 0:
        results[dataset] = []
        continue

    # normalize within dataset, using only the phrases that actually occur
    max_score = scores[nonzero_idx].max()
    if max_score > 0:
        scores = scores / max_score

    # rank only among phrases with a nonzero (real) score
    ranked = nonzero_idx[np.argsort(scores[nonzero_idx])[::-1]]
    top_idx = ranked[:10]

    results[dataset] = [
        (feature_names[i], scores[i]) for i in top_idx
    ]


In [11]:
rows = []

for dataset, phrases in results.items():
    for phrase, score in phrases:
        rows.append({
            "dataset": dataset,
            "phrase": phrase,
            "score": score
        })

plot_df = pd.DataFrame(rows)


In [12]:

dataset_order = ["ceri", "fora", "iq2", "umod", "whow", "wikitactics"]

# one color per dataset panel, drawn from the shared colorblind-safe palette
colors = COLORBLIND_PALETTE

# 2x3 GRID
fig, axes = plt.subplots(
    2, 3,
    constrained_layout=True,
)

axes = axes.flatten()

for i, dataset in enumerate(dataset_order):
    ax = axes[i]

    data = plot_df[plot_df["dataset"] == dataset] \
        .sort_values("score", ascending=True)

    # top phrases only (zero-score / non-occurring phrases were already
    # excluded upstream, so every bar shown here is a real occurrence)
    phrases = data["phrase"].tolist()
    scores = data["score"].tolist()

    # truncate softly
    def shorten(p):
        return p if len(p) <= 28 else p[:27] + "…"

    labels = [shorten(p) for p in phrases]

    ax.barh(
        labels,
        scores,
        color=colors[i],
        edgecolor="black",
    )

    ax.set_title(dataset)
    ax.set_xlim(0, 1.05)

    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.grid(axis='x', linestyle='--', alpha=0.2)

    ax.set_xlabel("")

fig.supxlabel(" ")

save_plot(Path("../graphs/tfidf_pefk.png"))


Figure saved to /home/dimits/Documents/research/interventions/graphs/tfidf_pefk.png
